In [ ]:
# imports
import io
import re
from pathlib import Path

import pandas as pd

In [ ]:
# folder with yearly files: 2014.csv ... 2024.csv
DATA_DIR = Path("./ranking")

In [ ]:
# load one tab-separated file, fix World Rank split across two lines (2017/2023/2024)
# standardize country column name, coerce Score to numeric, tag with Year
def load_year_file(path: Path) -> pd.DataFrame:
    raw_text = path.read_text(encoding="utf-8", errors="replace")

    wrapped_rank_pattern = re.compile(r"(\d+)\s*\nTop\s+[\d.]+%\t")
    clean_text = wrapped_rank_pattern.sub(r"\1\t", raw_text)

    df = pd.read_csv(io.StringIO(clean_text), sep="\t")

    df = df.rename(columns={"Country": "Location"})

    df["World Rank"] = df["World Rank"].astype(str).str.extract(r"(\d+)").astype("Int64")
    df["Score"] = pd.to_numeric(df["Score"], errors="coerce")

    year = int(re.search(r"(\d{4})", path.stem).group(1))
    df["Year"] = year

    return df

In [ ]:
# load all years, keep columns common to every year, concatenate
def build_consistent_dataframe(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {data_dir!r}")

    per_year_dfs = [load_year_file(f) for f in files]

    common_cols = set(per_year_dfs[0].columns)
    for df in per_year_dfs[1:]:
        common_cols &= set(df.columns)
    common_cols.discard("Year")

    ordered_common_cols = [c for c in per_year_dfs[0].columns if c in common_cols]
    final_cols = ordered_common_cols + ["Year"]

    combined = pd.concat([df[final_cols] for df in per_year_dfs], ignore_index=True)
    combined = combined.rename(columns={"Location": "Country"})

    return combined


df = build_consistent_dataframe(DATA_DIR)
df.head(10)

In [ ]:
# keep European (+ Turkey) institutions only
EUROPEAN_COUNTRIES = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary",
    "Iceland", "Ireland", "Italy", "Lithuania", "Luxembourg", "Netherlands",
    "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovak Republic",
    "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom", "Turkey",
}

df_europe = df[df["Country"].isin(EUROPEAN_COUNTRIES)].reset_index(drop=True)
df_europe.head(10)

In [ ]:
# rank institutions within Europe only, per year, by Score
df_europe["European Rank"] = (
    df_europe.groupby("Year")["Score"]
    .rank(ascending=False, method="min")
    .astype("Int64")
)

In [ ]:
# final output: institution, country, Year, European Rank only
df_final = (
    df_europe[["Institution", "Country", "Year", "European Rank"]]
    .sort_values(["Year", "European Rank"])
    .reset_index(drop=True)
)
df_final.head(10)